In [24]:
import numpy as np

from tqdm.auto import tqdm
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

## Q1. Embedding query

In [2]:
embed = Embedder()

In [11]:
query = "How does approximate nearest neighbor search work?"

query_vector = embed.encode(query)
query_vector[0]

np.float64(-0.020582036807885073)

### Question: The embedder returns a vector of 384 numbers. What's the first value (v[0])?
### Answer: -0.02

## Q2. Cosine similarity

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [13]:
sqlitesearch_document = next(document for document in documents if document["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
sqlitesearch_document

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [ ]:
sqlitesearch_vector = embed.encode(sqlitesearch_document["content"])
query_vector.dot(sqlitesearch_vector)

np.float64(0.3610702803026059)

### Question: Take the page `02-vector-search/lessons/07-sqlitesearch-vector.ml`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?
### Answer: 0.37

## Q3. Chunking and search by hand

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [13]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk["content"] for chunk in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

In [15]:
X = np.array(vectors)
scores = X.dot(query_vector)
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.648901732433228))

In [16]:
chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [17]:
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

### Question: Which file does the highest-scoring chunk belong to (its `filename`)?
### Answer: `02-vector-search/lessons/07-sqlitesearch-vector.md`

## Q4. Vector search with minsearch

In [19]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [20]:
query_2 = "What metric do we use to evaluate a search engine?"
query_2_vector = embed.encode(query_2)

In [28]:
results = vector_index.search(query_2_vector, num_results=1)
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [ ]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

### Question: Which file is the `filename` of the first result?
### Answer: `04-evaluation/lessons/05-search-metrics.md`

## Q5. Text search vs vector search

In [25]:
query_3 = "How do I store vectors in PostgreSQL?"
query_3_vector = embed.encode(query_3)

In [26]:
text_index = Index(text_fields=["content"])
text_index.fit(chunks)

In [31]:

text_results = text_index.search(query_3, num_results=5)
vector_results = vector_index.search(query_3_vector, num_results=5)
text_results[0], vector_results[0]

({'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [ ]:
text_set = set([chunk["filename"] for chunk in text_results])
vector_set = set([chunk["filename"] for chunk in vector_results])
text_set, vector_set

({'02-vector-search/lessons/01-intro.md',
  '02-vector-search/lessons/02-embeddings.md',
  '03-orchestration/lessons/05-rag.md'},
 {'02-vector-search/lessons/08-pgvector.md',
  '03-orchestration/lessons/05-rag.md'})

In [46]:
vector_set.difference(text_set)

{'02-vector-search/lessons/08-pgvector.md'}

### Question: Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
### Answer: `02-vector-search/lessons/08-pgvector.md`